In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings. Keep MCP calls serial; use the CLI through uv run for batch operations and final verification.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as read_nb, write_nb, update_cell, exec_nb, and diff_nb.
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import nbskill.mcp as _mcp_mod
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.mcp import capture_call as _example_capture_call
from nbskill.mcp import create_mcp as _example_create_mcp
from nbskill.read import read_nb as _example_read_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
def _demo_tool():
    print("captured output")

print(_example_capture_call(_demo_tool))
print(type(_example_create_mcp()).__name__)

captured output
FastMCP


In [ ]:
#| export
import os,json
import shutil
import sys
import threading
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

from fastcore.script import Param, call_parse, _in_call_parse
from fastmcp import FastMCP
from fastmcp.tools import ToolResult
from mcp.types import TextContent

from nbskill.convert import py2nb as _py2nb
from nbskill.convert import py2nbdev as _py2nbdev
from nbskill.convert import py2nbs as _py2nbs
from nbskill.edit_interactive import execute_plan as _execute_plan
from nbskill.edit_interactive import execute_project_plan as _execute_project_plan
from nbskill.execute import exec_nb as _exec_nb
from nbskill.graph import private_symbol_report as _private_symbol_report
from nbskill.graph import symbol_graph as _symbol_graph
from nbskill.parallel import notebook_locks
from nbskill.read import read_nb as _read_nb
from nbskill.read import show_doc as _show_doc
from nbskill.review import style_check as _style_check
from nbskill.review import style_report as _style_report
from nbskill.review import diff_nb as _diff_nb
from nbskill.write import batch_edit_nb as _batch_edit_nb
from nbskill.write import update_cell as _update_cell
from nbskill.write import write_nb as _write_nb

### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)


_CAPTURE_LOCK = threading.RLock()


def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"


def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    try:
        with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
            result = func(**kwargs)
    except SystemExit as exc:
        chunks = []
        if out.getvalue(): chunks.append(out.getvalue().rstrip())
        if err.getvalue(): chunks.append(err.getvalue().rstrip())
        chunks.append(f"SystemExit: {exc.code}")
        raise RuntimeError(chr(10).join(chunk for chunk in chunks if chunk)) from exc
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)


def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)


def _json_preview(value, limit=1200):
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    if len(text) <= limit: return text
    return f"{text[:limit].rstrip()}\n... truncated ..."


def _text_preview(value, limit=12000):
    text = as_text(value)
    if limit is None or len(text) <= limit:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - limit
    return {
        "text": f"{text[:limit].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }


def mcp_tool_result(tool, arguments, full_output, max_output_chars=12000, **structured):
    "Return visible MCP text plus structured data for clients that inspect it."
    call = {"tool": tool, "arguments": arguments}
    preview = _text_preview(full_output or "", limit=max_output_chars)
    summary = "\n".join([
        f"{tool} completed",
        "",
        "Call:",
        _json_preview(call),
        "",
        "Result:",
        preview["text"],
    ])
    data = {
        "summary": summary,
        "call": call,
        "full_output": preview["text"],
        "output_truncated": preview["truncated"],
        "output_chars": preview["chars"],
        "omitted_chars": preview["omitted_chars"],
    }
    data.update(structured)
    return ToolResult(
        content=[TextContent(type="text", text=summary)],
        structured_content=data,
    )


def _status_data():
    scripts = [
        "read_nb", "write_nb", "update_cell", "batch_edit_nb", "show_doc",
        "exec_nb", "diff_nb", "style_check", "install_nbskill",
        "symbol_graph", "private_symbol_report", "nbskill_mcp",
    ]
    return {
        "version": _package_version(),
        "cwd": str(Path.cwd()),
        "python": sys.executable,
        "mcp_command": "nbskill_mcp",
        "mcp_command_path": shutil.which("nbskill_mcp"),
        "cli_tools": {name: shutil.which(name) for name in scripts},
        "reconnect_hint": "Restart or reconnect the MCP client after reinstalling nbskill or changing tool signatures.",
        "install_commands": [
            "uv tool install --editable . --force",
            "codex mcp add nbskill -- nbskill_mcp",
            "claude mcp add nbskill -- nbskill_mcp",
        ],
    }


def _format_status(data):
    lines = [
        "nbskill status",
        f"version={data['version']}",
        f"cwd={data['cwd']}",
        f"python={data['python']}",
        f"mcp_command={data['mcp_command']}",
        f"mcp_command_path={data['mcp_command_path'] or '(not on PATH)'}",
        "cli_tools:",
    ]
    lines.extend(f"- {name}: {path or '(not on PATH)'}" for name, path in data["cli_tools"].items())
    lines.append(f"reconnect_hint={data['reconnect_hint']}")
    lines.append("install_commands:")
    lines.extend(f"- {cmd}" for cmd in data["install_commands"])
    return "\n".join(lines)


@call_parse
def nbskill_status(json_output: bool = False):  # Print JSON instead of text
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    data = _status_data()
    print(json.dumps(data, indent=2, sort_keys=True) if json_output else _format_status(data))
    return data if not _in_call_parse else None

In [ ]:
data = _status_data()
assert data["mcp_command"] == "nbskill_mcp"
assert "batch_edit_nb" in data["cli_tools"]

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

In [ ]:
#| export
def create_mcp():
    "Create the nbskill FastMCP server."
    capabilities = (
        "healthcheck,read_nb,show_doc,write_nb,update_cell,batch_edit_nb,exec_nb,diff_nb,"
        "execute_plan,execute_project_plan,symbol_graph,private_symbol_report,style_check,py2nb,py2nbs,py2nbdev"
    )
    mcp = FastMCP(
        "nbskill",
        instructions=(
            "Work notebook-first in nbdev projects. Prefer read_nb/show_doc for context, "
            "write_nb/update_cell/batch_edit_nb for edits, exec_nb for safe visible notebook execution, "
            "execute_plan for bounded interactive single-notebook work, "
            "and style_check for capped notebook hygiene reports. "
            "Notebook operations are concurrency-safe: calls touching the same notebook are serialized, "
            "calls touching different notebooks can run in parallel, and execution uses a global semaphore. "
            "Keep documentation before exported code and show-off examples after it."
        ),
    )

    @mcp.tool(name="healthcheck")
    def healthcheck_tool() -> ToolResult:
        "Return nbskill MCP status, installed version, process details, and available notebook tool capabilities."
        data = _status_data()
        full_output = "\n".join([
            "nbskill mcp ok",
            f"version={data['version']}",
            f"cwd={Path.cwd()}",
            f"python={sys.executable}",
            f"pid={os.getpid()}",
            f"capabilities={capabilities}",
            "parallel=same-notebook operations serialized; different notebooks may run in parallel",
            "execution=global semaphore with one active safe notebook execution",
            "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
        ])
        return mcp_tool_result("healthcheck", {}, full_output, status=data, capabilities=capabilities.split(","))

    @mcp.tool(name="read_nb")
    def read_nb_tool(path: str, query: str | None = None, cell_id: str | None = None, chapter: str | None = None, cell_type: str | None = None, contains: str | None = None, context: str = "overview", show_ids: bool = False) -> ToolResult:
        "Read notebook source without exposing raw JSON."
        arguments = dict(path=path, query=query, cell_id=cell_id, chapter=chapter, cell_type=cell_type, contains=contains, context=context, show_ids=show_ids)
        full_output = capture_notebook_call(_read_nb, path, path=path, query=query, cell_id=cell_id, chapter=chapter, cell_type=cell_type, contains=contains, context=context, show_ids=show_ids)
        return mcp_tool_result("read_nb", arguments, full_output)

    @mcp.tool(name="show_doc")
    def show_doc_tool(path: str, symbol: str, context: int = 2, source: bool = False, show_ids: bool = False) -> ToolResult:
        "Show the notebook story around one exported symbol."
        arguments = dict(path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)
        full_output = capture_notebook_call(_show_doc, path, path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)
        return mcp_tool_result("show_doc", arguments, full_output)

    @mcp.tool(name="write_nb")
    def write_nb_tool(
        path: str, cells: str = "", cells_file: str | None = None, before_id: str | None = None,
        after_id: str | None = None, chapter: str | None = None, replace: bool = False,
        cell_type: str = "code", export: bool = True, run_test: bool = False,
        run_style: bool = False, style_strict: bool = False, validate_code: bool = True,
        old_str: str | None = None, new_str: str | None = None, dry_run: bool = False,
        show_cells: bool = False,
    ) -> ToolResult:
        "Insert notebook cells or perform exact literal replacements across notebooks."
        arguments = dict(path=path, cells=cells, cells_file=cells_file, before_id=before_id, after_id=after_id, chapter=chapter, replace=replace, cell_type=cell_type, export=export, run_test=run_test, run_style=run_style, style_strict=style_strict, validate_code=validate_code, old_str=old_str, new_str=new_str, dry_run=dry_run, show_cells=show_cells)
        full_output = capture_notebook_call(_write_nb, path, **arguments)
        return mcp_tool_result("write_nb", arguments, full_output)

    @mcp.tool(name="update_cell")
    def update_cell_tool(
        path: str, new: str = "", new_file: str | None = None, cell_id: str | None = None,
        old_str: str | None = None, line_range: str | None = None, source_hash: str | None = None,
        cell_type: str = "code", export: bool = True, run_test: bool = False,
        validate_code: bool = True, dry_run: bool = False,
    ) -> ToolResult:
        "Update one existing notebook cell by id, old text, or line range."
        arguments = dict(path=path, new=new, new_file=new_file, cell_id=cell_id, old_str=old_str, line_range=line_range, source_hash=source_hash, cell_type=cell_type, export=export, run_test=run_test, validate_code=validate_code, dry_run=dry_run)
        full_output = capture_notebook_call(_update_cell, path, **arguments)
        return mcp_tool_result("update_cell", arguments, full_output)

    @mcp.tool(name="batch_edit_nb")
    def batch_edit_nb_tool(plan: str = "", plan_file: str | None = None, path: str | None = None, dry_run: bool = True, export: bool = True, validate_code: bool = True, default_cell_type: str = "code") -> ToolResult:
        "Apply a JSON batch edit plan to one or more notebooks."
        arguments = dict(plan=plan, plan_file=plan_file, path=path, dry_run=dry_run, export=export, validate_code=validate_code, default_cell_type=default_cell_type)
        full_output = capture_call(_batch_edit_nb, **arguments)
        return mcp_tool_result("batch_edit_nb", arguments, full_output)

    @mcp.tool(name="exec_nb")
    def exec_nb_tool(
        path: str, dest: str | None = None, exc_stop: bool = False, up2id: int | str | None = None,
        chapter: str | None = None, timeout: int = 30, show_output: bool = True,
        verbose: bool = False, safe: bool = True, allow: str | None = None,
        ok_dests: str | None = None, cache_httpx: bool = False, cache_dir: str | None = None,
        cache_domains: str | None = None, allow_new: bool = False,
    ) -> ToolResult:
        "Execute a notebook and return visible outputs/errors."
        arguments = dict(path=path, dest=dest, exc_stop=exc_stop, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output, verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains, allow_new=allow_new)
        full_output = capture_notebook_call(_exec_nb, path, dest or path, **arguments)
        return mcp_tool_result("exec_nb", arguments, full_output)

    @mcp.tool(name="diff_nb")
    def diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False) -> ToolResult:
        "Diff notebook code cells without expanding raw notebook JSON."
        arguments = dict(path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels)
        full_output = capture_notebook_call(_diff_nb, path, **arguments)
        return mcp_tool_result("diff_nb", arguments, full_output)

    @mcp.tool(name="execute_plan")
    def execute_plan_tool(notebook: str, plan: str, model: str | None = None, max_steps: int = 20, timeout: int = 30, export: bool = True, dry_run: bool = False) -> ToolResult:
        "Run a bounded edit-interactive loop against exactly one notebook."
        arguments = dict(notebook=notebook, plan=plan, model=model, max_steps=max_steps, timeout=timeout, export=export, dry_run=dry_run)
        full_output = capture_call(_execute_plan, **arguments)
        return mcp_tool_result("execute_plan", arguments, full_output)

    @mcp.tool(name="execute_project_plan")
    def execute_project_plan_tool(plan: str, notebooks: str | None = None, model: str | None = None, max_steps: int = 20, timeout: int = 30, export: bool = True, dry_run: bool = True) -> ToolResult:
        "Coordinate a broad plan as notebook-scoped execute_plan calls."
        arguments = dict(plan=plan, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, export=export, dry_run=dry_run)
        full_output = capture_call(_execute_project_plan, **arguments)
        return mcp_tool_result("execute_project_plan", arguments, full_output)

    @mcp.tool(name="symbol_graph")
    def symbol_graph_tool(path: str = "nbs", symbol: str = "") -> ToolResult:
        "Show definitions, callers, and callees for one notebook symbol."
        arguments = dict(path=path, symbol=symbol)
        full_output = capture_call(_symbol_graph, **arguments)
        return mcp_tool_result("symbol_graph", arguments, full_output)

    @mcp.tool(name="private_symbol_report")
    def private_symbol_report_tool(path: str = "nbs") -> ToolResult:
        "Report cross-notebook calls to private underscore-prefixed symbols."
        arguments = dict(path=path)
        full_output = capture_call(_private_symbol_report, **arguments)
        return mcp_tool_result("private_symbol_report", arguments, full_output)

    @mcp.tool(name="style_check")
    def style_check_tool(
        path: str = ".", skip_folder_re: str | None = None, skip_path: str | None = None,
        strict: bool = False, delete_after_output: bool = False, delete_after_outout: bool = False,
        max_output_chars: int = 12000, max_diagnostics: int = 200, fix: bool = False,
        dry_run: bool = True,
    ) -> ToolResult:
        "Print capped fast.ai style hints plus nbskill notebook hygiene warnings and global tool usage."
        arguments = dict(path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict, delete_after_output=delete_after_output, delete_after_outout=delete_after_outout, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, dry_run=dry_run)
        full_output = capture_call(_style_check, **arguments)
        report = _style_report(path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        return mcp_tool_result("style_check", arguments, full_output, max_output_chars=max_output_chars, style_report=report)

    @mcp.tool(name="py2nb")
    def py2nb_tool(
        path: str, nbs_path: str = "nbs", dest: str | None = None, recursive: bool = True,
        maxdepth: int | None = None, preserve_tree: bool = True, class_lines: int = 100,
        method_lines: int = 10, package: str | None = None, include: str | None = None,
        exclude: str | None = None, skip_init: bool = True, include_tests: bool = False,
        dry_run: bool = False, force: bool = True,
    ) -> ToolResult:
        "Convert one Python file or folder into nbdev notebook source."
        arguments = dict(path=path, nbs_path=nbs_path, dest=dest, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, dry_run=dry_run, force=force)
        full_output = capture_call(_py2nb, **arguments)
        return mcp_tool_result("py2nb", arguments, full_output)

    @mcp.tool(name="py2nbs")
    def py2nbs_tool(
        path: str, nbs_path: str = "nbs", recursive: bool = True, maxdepth: int | None = None,
        preserve_tree: bool = True, class_lines: int = 100, method_lines: int = 10,
        package: str | None = None, include: str | None = None, exclude: str | None = None,
        skip_init: bool = True, include_tests: bool = False, dry_run: bool = False,
        force: bool = True,
    ) -> ToolResult:
        "Convert Python files in a folder into valid nbdev notebooks."
        arguments = dict(path=path, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, dry_run=dry_run, force=force)
        full_output = capture_call(_py2nbs, **arguments)
        return mcp_tool_result("py2nbs", arguments, full_output)

    @mcp.tool(name="py2nbdev")
    def py2nbdev_tool(source: str, dest: str, package: str | None = None, nbs_path: str = "nbs", dry_run: bool = True, force: bool = False, run_validation: bool = True) -> ToolResult:
        "Create a pragmatic nbdev project from a pure-Python package."
        arguments = dict(source=source, dest=dest, package=package, nbs_path=nbs_path, dry_run=dry_run, force=force, run_validation=run_validation)
        full_output = capture_call(_py2nbdev, **arguments)
        return mcp_tool_result("py2nbdev", arguments, full_output)

    return mcp

### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
@call_parse
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    create_mcp().run(transport=transport, show_banner=show_banner)

In [ ]:
mcp = create_mcp()
tools = {tool.name: tool for tool in await mcp.list_tools()}
assert {
    "healthcheck", "read_nb", "write_nb", "update_cell", "batch_edit_nb",
    "exec_nb", "show_doc", "execute_plan", "execute_project_plan",
    "symbol_graph", "private_symbol_report", "style_check", "py2nb",
    "py2nbs", "py2nbdev",
} <= set(tools)
assert "cells_file" in str(tools["write_nb"].parameters)
assert "new_file" in str(tools["update_cell"].parameters)
assert "dry_run" in str(tools["execute_plan"].parameters)
assert "max_output_chars" in str(tools["style_check"].parameters)

In [ ]:
calls = {}
old_execute_plan = _mcp_mod._execute_plan

try:
    def fake_execute_plan(**kwargs):
        calls.update(kwargs)
        return "delegated"

    _mcp_mod._execute_plan = fake_execute_plan
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "execute_plan",
        {
            "notebook": "nbs/index.ipynb",
            "plan": "noop",
            "model": "fake",
            "max_steps": 1,
            "timeout": 2,
            "export": False,
            "dry_run": True,
        },
    )
    assert calls == {
        "notebook": "nbs/index.ipynb",
        "plan": "noop",
        "model": "fake",
        "max_steps": 1,
        "timeout": 2,
        "export": False,
        "dry_run": True,
    }
    assert "delegated" in str(result)
finally:
    _mcp_mod._execute_plan = old_execute_plan

In [ ]:
path = demo_path("07_mcp_sample.ipynb")
_write_nb(path,new_nb([mk_cell("#| default_exp sample", cell_type="code")]))
text = capture_call(_read_nb, path=str(path), context="overview")
assert "default_exp sample" in text
remove_demo_path(path)

Wrote 1 cells to nbs/data/07_mcp_sample.ipynb and exported with nbdev


Path('nbs/data/07_mcp_sample.ipynb')

In [ ]:
assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

def _prints_and_returns():
    print("printed")
    return "returned"

assert capture_call(_prints_and_returns) == "printed"

def _prints_and_exits():
    print("before exit")
    raise SystemExit(7)

try:
    capture_call(_prints_and_exits)
except RuntimeError as exc:
    assert "before exit" in str(exc)
    assert "SystemExit: 7" in str(exc)
else:
    raise AssertionError("SystemExit should be converted to RuntimeError for MCP tools")

import time

original_stdout = sys.stdout
outputs = []
errors = []
entered = threading.Event()

def _slow_print(label, delay, signal=None):
    def inner():
        if signal is not None: signal.set()
        time.sleep(delay)
        print(label)
    return inner

def _capture_worker(label, delay, signal=None):
    try:
        outputs.append(capture_call(_slow_print(label, delay, signal=signal)))
    except BaseException as exc:
        errors.append(exc)

threads = [
    threading.Thread(target=_capture_worker, args=("first", 0.03, entered)),
    threading.Thread(target=_capture_worker, args=("second", 0.01)),
]
threads[0].start()
assert entered.wait(1)
threads[1].start()
for thread in threads: thread.join()

assert errors == []
assert sorted(outputs) == ["first", "second"]
assert sys.stdout is original_stdout